# 11. Feedback models

## Numerical experiments - Week 38/2025

_Boyan Mihaylov, MSc Computational Science (UVA/VU)_

The collection of models can be further expanded by taking into account mechanisms of simultaneous two-sided influence by the inducer and the inhibitor. First, this can be used to correct the flaw in the assumption of an inducer-controlled inhibitor permeability. If the inducer causes a change in the cell wall porosity, then this would affect its own permeation with an equivalent factor. Second, feedback loops between the two actors can be explored, resulting in models with:
- inhibitor-dependent induction strength AND inducer-dependent inhibitor permeability;
- inhibitor-dependent induction strength AND inducer-dependent inhibitor threshold;
- inhibitor-dependent induction threshold AND inducer-dependent inhibitor permeability;
- inhibitor-dependent induction threshold AND inducer-dependent inhibitor threshold.

Each of these compound interaction can be combined with either a single-factor (inducer- or inhibitor-defined) germination or a two-factor germination, resulting in a total of 12 additional models.

## Prerequisite libraries

In [1]:
using PyPlot

include("../src/conversions.jl")
include("../src/diffusion.jl")
include("../src/setup.jl")
include("../src/plotting.jl")
include("../src/analysis.jl")
include("../src/datautils.jl")
include("../src/germstats.jl")

using .Conversions
using .Diffusion
using .Setup
using .Plotting
using .Analysis
using .DataUtils
using .GermStats

## 1. Inducer-dependent permeability - revision

The change of inhibitor concentration in the spore over time is given by the ODE

$$
\begin{equation}
\frac{d{c_{\textrm{in}}^{\textrm{I}}}}{d{t}}=-\left(\frac{(1+s) P_s^{\textrm{I}}A}{V_s}\right)\left[c_{\textrm{in}}^{\textrm{I}}{(t)}-c_{\textrm{out}}^{\textrm{I}}{(t)}\right],
\end{equation}
$$

where the signal strength $s$ modulates the inhibitor permeation through the cell wall. This happens via a linear shift from a base permeability value, since it is assumed that inhibitor molecules can still escape the cell wall in the complete absence of induction.

The outside concentration changes like

$$
\begin{equation}
\frac{d{c_{\textrm{out}}^{\textrm{I}}}}{d{t}}=\left(\frac{(1+s) P_s^{\textrm{I}}A}{V_{\textrm{free}}}\right)\left[c_{\textrm{in}}^{\textrm{I}}{(t)}-c_{\textrm{out}}^{\textrm{I}}{(t)}\right],
\end{equation}
$$

where $V_{\textrm{free}}=V_{\textrm{des}}-V_s$ is the volume unoccupied by a spore in the equally designated space per spore.

The concentration drop of inducer across the outer cell wall, on the other hand, follows the formula

$$
\begin{equation}
\frac{d{\Delta c^{\textrm{C}}}}{d{t}}=-\left(\frac{(1+s)P_s^{\textrm{C}}A}{V_s}\right)\left[c_{\textrm{in}}^{\textrm{C}}{(t)}-c_{\textrm{out}}^{\textrm{C}}{(t)}\right].
\end{equation}
$$

The permeability is modulated via the cell wall structure, so the scaling factor $s$ is used here as well. Since $\Delta c^{\textrm{C}}=c_{\textrm{in}}^{\textrm{C}}{(t)}-c_{\textrm{out}}^{\textrm{C}}{(t)}$ and $c_{\textrm{out}}^{\textrm{C}}{(t)}=c_{\textrm{ex}}$ is a constant, this equates to

$$
\begin{equation}
\frac{d{c_{\textrm{in}}^{\textrm{C}}}}{d{t}}=-\left(\frac{(1+s)P_s^{\textrm{C}}A}{V_s}\right)\left[c_{\textrm{in}}^{\textrm{C}}{(t)}-c_{\textrm{ex}}\right].
\end{equation}
$$

As a reminder,

$$
\begin{equation}
    s{(c_{\textrm{in}}^{\textrm{C}})}=s_{\textrm{max}}\frac{c_{\textrm{in}}^{\textrm{C}}}{K_{\textrm{cs}}+c_{\textrm{in}}^{\textrm{C}}},
\end{equation}
$$

where $s_{\textrm{max}}$ scales the concentration function to its appropriate magnitude of effect on the permeability. Therefore,

$$
\begin{equation}
    1+s{(c_{\textrm{in}}^{\textrm{C}})}=\frac{K_{\textrm{cs}}+c_{\textrm{in}}^{\textrm{C}}(1+s_{\textrm{max}})}{K_{\textrm{cs}}+c_{\textrm{in}}^{\textrm{C}}}.
\end{equation}
$$

Thus, the following system of coupled ODEs needs to be solved:

$$
\begin{equation}
\dot{c}_{\textrm{in}}^{\textrm{I}}=-\left[\frac{(K_{\textrm{cs}}+c_{\textrm{in}}^{\textrm{C}}(1+s_{\textrm{max}}))\\\ P_s^{\textrm{I}}A}{V_s(K_{\textrm{cs}}+c_{\textrm{in}}^{\textrm{C}})}\right]\left[c_{\textrm{in}}^{\textrm{I}}{(t)}-c_{\textrm{out}}^{\textrm{I}}{(t)}\right]
\end{equation}
$$

$$
\begin{equation}
\dot{c}_{\textrm{out}}^{\textrm{I}}=\left[\frac{(K_{\textrm{cs}}+c_{\textrm{in}}^{\textrm{C}}(1+s_{\textrm{max}}))\\\ P_s^{\textrm{I}}A}{\left(1/\rho_s-V_s\right)(K_{\textrm{cs}}+c_{\textrm{in}}^{\textrm{C}})}\right]\left[c_{\textrm{in}}^{\textrm{I}}{(t)}-c_{\textrm{out}}^{\textrm{I}}{(t)}\right],
\end{equation}
$$

$$
\begin{equation}
\dot{c}_{\textrm{in}}^{\textrm{C}}=-\left[\frac{(K_{\textrm{cs}}+c_{\textrm{in}}^{\textrm{C}}(1+s_{\textrm{max}}))\\\ P_s^{\textrm{C}}A}{V_s(K_{\textrm{cs}}+c_{\textrm{in}}^{\textrm{C}})}\right]\left[c_{\textrm{in}}^{\textrm{C}}{(t)}-c_{\textrm{ex}}\right].
\end{equation}
$$

Factoring out $g(c_{\textrm{in}}^{\textrm{C}})=\frac{(K_{\textrm{cs}}+c_{\textrm{in}}^{\textrm{C}}(1+s_{\textrm{max}}))\\\ A}{K_{\textrm{cs}}+c_{\textrm{in}}^{\textrm{C}}}$,

$$
\begin{equation}
\dot{c}_{\textrm{in}}^{\textrm{I}}=-\frac{g(c_{\textrm{in}}^{\textrm{C}}) P_s^{\textrm{I}}}{V_s}\left[c_{\textrm{in}}^{\textrm{I}}{(t)}-c_{\textrm{out}}^{\textrm{I}}{(t)}\right]
\end{equation}
$$

$$
\begin{equation}
\dot{c}_{\textrm{out}}^{\textrm{I}}=\frac{g(c_{\textrm{in}}^{\textrm{C}}) P_s^{\textrm{I}}}{1/\rho_s-V_s}\left[c_{\textrm{in}}^{\textrm{I}}{(t)}-c_{\textrm{out}}^{\textrm{I}}{(t)}\right],
\end{equation}
$$

$$
\begin{equation}
\dot{c}_{\textrm{in}}^{\textrm{C}}=-\frac{g(c_{\textrm{in}}^{\textrm{C}}) P_s^{\textrm{C}}}{V_s}\left[c_{\textrm{in}}^{\textrm{C}}(t)-c_{\textrm{ex}}\right].
\end{equation}
$$

The integration of these ODEs is demonstrated below. As a check, the number of inhibitor molecules is tracked by referring the concentrations to the respective volumes.